In [ ]:
# Data sources:
# For the modal splits, we use the following data sources:
# Enquête sur les comportements de Déplacements, 2024 edition (ECD7)
# https://data-mobility.irisnet.be/fr/info/b6b1de33-3fd7-4867-b519-25aa3df040e0/

In [ ]:
# Steps:
#   1. Load full population + HTS
#   2. Compute straight-line commute distance with detour factor
#   3. Build (has_car × distance_bin) modal split table from HTS,
#      rescale to match 2024 aggregate modal split
#   4. Assign mode by sampling from rescaled cell distributions
#   5. Fit single weighted KDE on HTS departure times (home -> work)
#   6. Fit single weighted KDE on HTS work durations
#   7. Sample departure time and work duration per agent
#   8. Derive departure work→home = departure + travel_time + duration
#   9. Write all new columns back to population CSV

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")

RNG = np.random.default_rng(42)

In [ ]:
# Configuration

# Distance bins (km) — asymmetric: no-car uses 2 bins, has-car uses 3
# We encode this as 3 bins and handle no-car/>10 as one cell
DIST_BINS   = [0, 5, 10, np.inf]
DIST_LABELS = ["<=5 km", "5-10 km", ">10 km"]
DETOUR_FACTOR = 1.3

# 2024 modal split (raw, will be normalised)
MODAL_2024_RAW = {"car": 29.1, "pt": 40.3, "bike": 14.6, "walk": 10.8}
total = sum(MODAL_2024_RAW.values())
MODAL_2024 = {m: v / total for m, v in MODAL_2024_RAW.items()}
MODES = ["car", "pt", "bike", "walk"]

print("2024 normalised modal split:")
for m, v in MODAL_2024.items():
    print(f"{m:<6} {v*100:.1f}%")

# Mode speeds (km/h) for travel time estimation
SPEEDS = {"car": 30, "pt": 20, "bike": 15, "walk": 5}

In [ ]:
# Load data
# Full population (not including teleworkers)
synpop = pd.read_csv("../6_teleworking/output/workers_not_teleworking.csv")
print(f"Population: {len(synpop):,} agents")

# HTS 2017
hts = pd.read_csv("input_data/hts_brussels.csv")
df_weights = pd.read_excel("input_data/weights_individuals.xlsx")
hts = hts.merge(df_weights, left_on='participant_id', right_on='p_id', how='left')
hts = hts.rename(columns={"WeightPOP": "weight"})
hts = hts.dropna(subset=["distance_km", "weight", "mode", "has_car"])
hts = hts[hts["distance_km"] > 0]
hts["has_car"] = hts["has_car"].astype(bool)
print(f"HTS records: {len(hts)}")

# TRANSPORT MODE

In [ ]:
# Compute approximate commute distances and brackets for synthetic population
synpop["commute_dist_km"] = (
    np.sqrt(
        (synpop["home_x"] - synpop["work_x"])**2 +
        (synpop["home_y"] - synpop["work_y"])**2
    ) / 1000 * DETOUR_FACTOR
)

synpop["dist_bracket"] = pd.cut(
    synpop["commute_dist_km"],
    bins=DIST_BINS,
    labels=DIST_LABELS,
    right=True,
    include_lowest=True
)

print(f"Distance distribution:")
print(synpop["dist_bracket"].value_counts().sort_index().to_string())
print(f"Min: {synpop['commute_dist_km'].min():.1f} km")
print(f"Max: {synpop['commute_dist_km'].max():.1f} km")
print(f"Mean: {synpop['commute_dist_km'].mean():.1f} km")
print(f"Median: {synpop['commute_dist_km'].median():.1f} km")

In [ ]:
# Build HTS modal split table
# Apply same binning to HTS, but no-car uses only 2 bins (collapse <=5 and 5-10 into <=10)
hts["dist_bracket"] = pd.cut(
    hts["distance_km"],
    bins=DIST_BINS,
    labels=DIST_LABELS,
    right=True,
    include_lowest=True
)

# For no-car group: merge <=5 and 5-10 into a single <=10 bin
def get_hts_cell(has_car, dist_bracket):
    if not has_car:
        return (False, "<=10 km") if dist_bracket in ["<=5 km", "5-10 km"] else (False, ">10 km")
    return (has_car, dist_bracket)

hts["cell"] = hts.apply(
    lambda r: get_hts_cell(r["has_car"], r["dist_bracket"]), axis=1)

# Weighted modal split per cell
cells = {}
for cell_key, cell_df in hts.groupby("cell", observed=True):
    total_w = cell_df["weight"].sum()
    split = {}
    for m in MODES:
        w = cell_df[cell_df["mode"] == m]["weight"].sum()
        split[m] = w / total_w if total_w > 0 else 0.0
    cells[cell_key] = {"split": split, "n": len(cell_df)}

print("\nRaw HTS modal splits per cell:")
print(f"{'Cell':<35} {'n':>4}  " + "".join(f"{m:>6}" for m in MODES))
print(f"{'─'*75}")
for k, v in sorted(cells.items()):
    splits_str = "".join(f"{v['split'][m]*100:>5.1f}%" for m in MODES)
    print(f"{str(k):<35} {v['n']:>4}  {splits_str}")

In [ ]:
# Rescale HTS splits to match 2024 aggregate modal split
# IPF-style rescaling: multiply each cell's mode share by (2024_share / hts_aggregate_share)
# First compute HTS aggregate weighted modal split
hts_total_w = hts["weight"].sum()
hts_agg = {m: hts[hts["mode"] == m]["weight"].sum() / hts_total_w for m in MODES}

print("HTS 2017 aggregate modal split:")
for m in MODES:
    print(f"{m:<6} {hts_agg[m]*100:.1f}%")

# Correction factors per mode
correction = {m: MODAL_2024[m] / hts_agg[m] if hts_agg[m] > 0 else 1.0 for m in MODES}
correction['car'] = 0.95  # Adjust car correction manually since we had too low car share
print("\nCorrection factors (2024 / 2017):")
for m in MODES:
    print(f"{m:<6} {correction[m]:.3f}")

# Apply correction and renormalise each cell (preserve zeros)
cells_rescaled = {}
for cell_key, cell_data in cells.items():
    raw = cell_data["split"]
    corrected = {m: raw[m] * correction[m] for m in MODES}
    total_c = sum(corrected.values())
    if total_c > 0:
        normalised = {m: corrected[m] / total_c for m in MODES}
    else:
        normalised = {m: MODAL_2024[m] for m in MODES}
    cells_rescaled[cell_key] = normalised

print("\nRescaled modal splits per cell:")
print(f"{'Cell':<35} {'n':>4}  " + "".join(f"{m:>6}" for m in MODES))
print(f"{'─'*75}")
for k, split in sorted(cells_rescaled.items()):
    n = cells[k]["n"]
    splits_str = "".join(f"{split[m]*100:>5.1f}%" for m in MODES)
    print(f"{str(k):<35} {n:>4}  {splits_str}")

In [ ]:
# Assign mode to each agent
def get_agent_cell(has_car, dist_bracket):
    if not has_car:
        return (False, "<=10 km") if dist_bracket in ["<=5 km", "5-10 km"] else (False, ">10 km")
    return (has_car, str(dist_bracket))

def sample_mode(cell_key, rng):
    if cell_key not in cells_rescaled:
        # Fallback: use 2024 aggregate split
        probs = [MODAL_2024[m] for m in MODES]
    else:
        probs = [cells_rescaled[cell_key][m] for m in MODES]
    return rng.choice(MODES, p=probs)

synpop["agent_cell"] = synpop.apply(
    lambda r: get_agent_cell(r["has_car"], r["dist_bracket"]), axis=1)

synpop["assigned_mode"] = synpop["agent_cell"].apply(
    lambda c: sample_mode(c, RNG))

print("Assigned modal split:")
mode_counts = synpop["assigned_mode"].value_counts()
for m in MODES:
    n = mode_counts.get(m, 0)
    pct = n / len(synpop) * 100
    print(f"{m:<6} {pct:.1f}%  (n={n:,})")

# DEPARTURE TIME + DURATION

In [ ]:
# Fit KDE on HTS departure times
hts["dep_minutes"] = hts["departure_home_to_work"]
hts_dep = hts.dropna(subset=["dep_minutes"])

print(f"Valid departure records: {len(hts_dep)}")
print(f"Mean departure: {int(hts_dep['dep_minutes'].mean())//60:02d}:{int(hts_dep['dep_minutes'].mean())%60:02d}")
print(f"Range: {int(hts_dep['dep_minutes'].min())//60:02d}:{int(hts_dep['dep_minutes'].min())%60:02d} - "
      f"{int(hts_dep['dep_minutes'].max())//60:02d}:{int(hts_dep['dep_minutes'].max())%60:02d}")

# Fit weighted KDE
dep_values  = hts_dep["dep_minutes"].values
dep_weights = hts_dep["weight"].values
dep_weights = dep_weights / dep_weights.sum()  # normalise
kde_departure = gaussian_kde(dep_values, weights=dep_weights, bw_method="scott")

# Sample from KDE — draw candidates and clip to observed range
DEP_MIN, DEP_MAX = 270, 660  # cap at 11:00

def sample_kde(kde, n, lo, hi, rng, max_iter=10):
    """Sample n values from KDE, clipped to [lo, hi]."""
    samples = []
    while len(samples) < n:
        candidates = kde.resample(n * 2, seed=int(rng.integers(1e6)))[0]
        valid = candidates[(candidates >= lo) & (candidates <= hi)]
        samples.extend(valid.tolist())
    return np.array(samples[:n])

In [ ]:
# Fit KDE on work duration
hts["arr_minutes"] = hts["arrival_work"]
hts["dep_work_minutes"] = hts["departure_from_work"]
hts["work_duration"] = hts["dep_work_minutes"] - hts["arr_minutes"]

# Keep only plausible durations (1h - 14h)
hts_dur = hts.dropna(subset=["work_duration"])
hts_dur = hts_dur[(hts_dur["work_duration"] >= 60) &
                   (hts_dur["work_duration"] <= 840)]

print(f"Valid duration records: {len(hts_dur)}")
print(f"Mean duration: {hts_dur['work_duration'].mean()/60:.1f}h")
print(f"Median duration: {hts_dur['work_duration'].median()/60:.1f}h")
print(f"Range: {hts_dur['work_duration'].min()/60:.1f}h - {hts_dur['work_duration'].max()/60:.1f}h")

dur_values  = hts_dur["work_duration"].values
dur_weights = hts_dur["weight"].values
dur_weights = dur_weights / dur_weights.sum()
kde_duration = gaussian_kde(dur_values, weights=dur_weights, bw_method="scott")

DUR_MIN, DUR_MAX = 60, 840

In [ ]:
# Sample times per agent
n_agents = len(synpop)
dep_samples = sample_kde(kde_departure, n_agents, DEP_MIN, DEP_MAX, RNG)
dur_samples = sample_kde(kde_duration,  n_agents, DUR_MIN, DUR_MAX, RNG)

synpop["departure_home_to_work"] = np.round(dep_samples).astype(int)
synpop["work_duration"]  = np.round(dur_samples).astype(int)

# Travel time (minutes) = distance / speed * 60
synpop["travel_time_min"] = (
    synpop["commute_dist_km"] /
    synpop["assigned_mode"].map(SPEEDS) * 60
).round().astype(int)

# Arrival at work
synpop["arrival_work"] = synpop["departure_home_to_work"] + synpop["travel_time_min"]

# Departure from work
synpop["departure_from_work"] = (
    synpop["arrival_work"] + synpop["work_duration"]
)

# Convert minutes to HH:MM strings
def minutes_to_hhmm(m):
    h = int(m) // 60
    mn = int(m) % 60
    return f"{h:02d}:{mn:02d}:00"

synpop["departure_home_time"] = synpop["departure_home_to_work"].apply(minutes_to_hhmm)
synpop["arrival_work_time"]   = synpop["arrival_work"].apply(minutes_to_hhmm)
synpop["departure_work_time"] = synpop["departure_from_work"].apply(minutes_to_hhmm)

In [ ]:
# Sanity checks
print("\n" + "═"*60)
print("SANITY CHECKS")
print("═"*60)

print(f"\nAll (n={len(synpop):,}):")
print(f"Departure home — mean: {minutes_to_hhmm(synpop['departure_home_to_work'].mean())}  "
      f"median: {minutes_to_hhmm(synpop['departure_home_to_work'].median())}")
print(f"Arrival work   — mean: {minutes_to_hhmm(synpop['arrival_work'].mean())}")
print(f"Work duration  — mean: {synpop['work_duration'].mean()/60:.1f}h  "
      f"median: {synpop['work_duration'].median()/60:.1f}h")
print(f"Departure work — mean: {minutes_to_hhmm(synpop['departure_from_work'].mean())}  "
      f"median: {minutes_to_hhmm(synpop['departure_from_work'].median())}")

print(f"\nModal split (all):")
mc = synpop["assigned_mode"].value_counts()
for m in MODES:
    n = mc.get(m, 0)
    print(f"{m:<6} {n/len(synpop)*100:.1f}%")

# Flag implausible plans
n_late_arrival = (synpop["arrival_work"] > 13*60).sum()
n_midnight_return = (synpop["departure_from_work"] > 24*60).sum()
print(f"\nImplausible plans:")
print(f"Arriving after 13:00: {n_late_arrival} ({n_late_arrival/len(synpop)*100:.1f}%)")
print(f"Departing work after midnight: {n_midnight_return} ({n_midnight_return/len(synpop)*100:.1f}%)")

## Departure times to work hourly brackets

In [ ]:
# Check the hourly distribution of departure times to be compared with the EFM (2025) HTS
testing_pop = synpop["departure_home_to_work"].copy()

testing_pop = testing_pop // 60 + 1

testing_pop.value_counts(normalize=True) * 100

In [ ]:
# Save
out_cols = [
    "commute_dist_km", "dist_bracket",
    "assigned_mode",
    "departure_home_time", "departure_home_to_work",
    "arrival_work_time",   "arrival_work",
    "work_duration",
    "departure_work_time", "departure_from_work",
    "travel_time_min",
]
output_path = "output/synpop_with_activity_plans.csv"
synpop.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(f"Columns added: {out_cols}")
print(f"Total agents: {len(synpop):,}")
print(f"\nDone.")

# Save
output_path = "output/workers_with_activity.csv"
synpop.to_csv(output_path, index=False)
print(f"Saved: {output_path}")